# Bulk upload artifacts from Excel or CSV

Load many artifacts, such as validation issues, limitations or policy exceptions, into ValidMind at once from a spreadsheet, instead of logging each one by hand in the ValidMind Platform.

This notebook reads an Excel (`.xlsx`) or CSV file with one artifact per row, checks every row against your organization, and then creates the artifacts through the ValidMind REST API. You get a result for every row (created, skipped, or failed with the reason) and a file of failed rows you can fix and upload again.

::: {.content-hidden when-format="html"}
## Contents
- [Introduction](#toc1__)
- [Configure your upload](#toc2__)
- [About ValidMind](#toc3__)
  - [Before you begin](#toc3_1__)
  - [New to ValidMind?](#toc3_2__)
  - [Key concepts](#toc3_3__)
- [Setting up](#toc4__)
  - [Install the required packages](#toc4_1__)
  - [Connect to the ValidMind API](#toc4_2__)
- [Choose what to upload](#toc5__)
  - [Choose the artifact type](#toc5_1__)
  - [Review the fields for this artifact type](#toc5_2__)
- [Prepare your file](#toc6__)
  - [Load your file](#toc6_1__)
  - [Map columns to fields](#toc6_2__)
  - [Convert rows into artifacts](#toc6_3__)
- [Check your file with a dry run](#toc7__)
- [Upload your artifacts](#toc8__)
  - [Review the results](#toc8_1__)
- [In summary](#toc9__)
- [Next steps](#toc10__)
  - [Discover more learning resources](#toc10_1__)

:::
<!-- jn-toc-notebook-config
	numbering=false
	anchor=true
	flat=false
	minLevel=2
	maxLevel=4
	/jn-toc-notebook-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

<a id='toc1__'></a>

## Introduction

Logging artifacts one at a time works for a few findings, but not when you are bringing hundreds of issues, limitations or exceptions from another tool into ValidMind. This notebook automates that:

1. You describe each artifact in a spreadsheet row: the record it belongs to, its title and description, and any other fields such as severity, assignee, due date or custom fields your organization uses.
2. A **dry run** checks every row without creating anything, so you can fix problems first.
3. The **upload** creates one artifact per row and reports what happened to each.

Uploading is safe to repeat. A row whose record already has an artifact of the same type with the same title is **skipped**, not duplicated, so after fixing failed rows you can run the same file again.

<a id='toc2__'></a>

## Configure your upload

These are the only settings you need to change. Set them here, then run the notebook from the top.

Start with `DRY_RUN = True`: the notebook checks every row and creates nothing. When the dry run looks right, set `DRY_RUN = False` and run the notebook again to upload.

In [ ]:
# The Excel (.xlsx) or CSV file to upload, one artifact per row.
FILE_PATH = "sample_artifacts.csv"

# True: check every row and create nothing. False: create the artifacts.
DRY_RUN = True

# The artifact type every row becomes, as a name or slug from "Choose the artifact type".
ARTIFACT_TYPE = "validation-issue"

# Rename columns whose headers do not match a field, for example:
# {"Model Name": "record", "Issue": "title", "Details": "description", "Owner": "assignee"}
COLUMN_MAP = {}

# True if text dates in your file put the day first (02/03/2024 is 2 March 2024).
DATES_DAY_FIRST = False

# True: do not email anyone about each new artifact.
SKIP_EMAIL_NOTIFICATIONS = True

<a id='toc3__'></a>

## About ValidMind

ValidMind is a suite of tools for managing risk, including risk associated with AI and statistical models. 

You use the ValidMind Library to automate documentation and validation tests, and then use the ValidMind Platform to collaborate on documentation. Together, these products simplify risk management, facilitate compliance with regulations and institutional standards, and enhance collaboration between yourself and validators.

<a id='toc3_1__'></a>

### Before you begin

This notebook assumes you have basic familiarity with Python, including an understanding of how functions work. If you are new to Python, you can still run the notebook but we recommend further familiarizing yourself with the language. 

If you encounter errors due to missing modules in your Python environment, install the modules with `pip install`, and then re-run the notebook. For more help, refer to [Installing Python Modules](https://docs.python.org/3/installing/index.html).

You also need:
- **API credentials** — your ValidMind API key and secret. To find them, follow [Get your API key and secret](https://docs.validmind.ai/reference/validmind-rest-api-vm.html#get-your-api-key-and-secret).
- **Permission to add artifacts** to the records in your file. Artifacts are created as the user who owns the API key.

<a id='toc3_2__'></a>

### New to ValidMind?

If you haven't already seen our documentation on the [ValidMind Library](https://docs.validmind.ai/developer/validmind-library.html), we recommend you begin by exploring the available resources in this section. There, you can learn more about documenting records such as models and running tests, as well as find code samples and our Python Library API reference.

<div class="alert alert-block alert-info" style="background-color: #B5B5B510; color: black; border: 1px solid #083E44; border-left-width: 5px; box-shadow: 2px 2px 4px rgba(0, 0, 0, 0.2);border-radius: 5px;"><span style="color: #083E44;"><b>For access to all features available in this notebook, you'll need access to a ValidMind account.</b></span>
<br></br>
<a href="https://docs.validmind.ai/guide/access/register-with-validmind.html" style="color: #DE257E;"><b>Register with ValidMind</b></a></div>

<a id='toc3_3__'></a>

### Key concepts

**record**: A tool tracked in the ValidMind inventory, such as a model. Every artifact belongs to one record.

**artifact**: An item logged against a record, such as a validation issue, a model limitation or a policy exception. Artifacts have a title, a description, a status and, depending on their type, a severity, an assignee, a due date and custom fields.

**artifact type**: The kind of artifact, such as a validation issue or a policy exception. Each artifact type has its own statuses, can turn built-in fields such as severity on or off, and can have its own custom fields.

**custom field**: A field your organization adds to its artifacts, such as a root cause or a list of affected data sources.

**ValidMind REST API**: The HTTP API this notebook calls to read your organization's settings and create artifacts. (Learn more: [ValidMind REST API](https://docs.validmind.ai/reference/validmind-rest-api-vm.html))

<a id='toc4__'></a>

## Setting up

<a id='toc4_1__'></a>

### Install the required packages

This notebook only needs `pandas` to read your file, `openpyxl` to read Excel files, `requests` to call the API, and `python-dotenv` to load your credentials. It does not need the ValidMind Library itself.

In [ ]:
%pip install -q pandas openpyxl requests python-dotenv

<a id='toc4_2__'></a>

### Connect to the ValidMind API

Store your credentials in a `.env` file next to this notebook so they stay out of the notebook itself:

```bash
VM_API_KEY=<your-api-key>
VM_API_SECRET=<your-api-secret>
# Only if your organization does not use the default ValidMind host:
# VM_PUBLIC_API_HOST=https://app.<your-region>.validmind.ai/vm/api/v1
```

The REST API host is different from the `api_host` you pass to `vm.init()` in other notebooks, which is why it has its own variable.

In [ ]:
%load_ext dotenv
%dotenv .env

import os
import re

import pandas as pd
import requests

pd.set_option("display.max_colwidth", None)

API_HOST = os.getenv("VM_PUBLIC_API_HOST", "https://app.prod.validmind.ai/vm/api/v1").rstrip("/")

session = requests.Session()
session.headers.update(
    {
        "accept": "application/json",
        "x-api-key": os.environ["VM_API_KEY"],
        "x-api-secret": os.environ["VM_API_SECRET"],
    }
)


def api(method, path, **kwargs):
    """Call the ValidMind REST API.

    Redirects are refused: following one turns a POST into a GET and drops its body.
    """
    response = session.request(method, f"{API_HOST}{path}", timeout=60, allow_redirects=False, **kwargs)
    if response.is_redirect:
        raise RuntimeError(
            f"{API_HOST} redirects to {response.headers['location']}. "
            "Set VM_PUBLIC_API_HOST to the host it redirects to."
        )
    return response


def api_error(response):
    """Return (code, message) from a failed API response."""
    try:
        error = response.json().get("error") or {}
    except ValueError:  # not JSON, such as a gateway error page
        return None, f"HTTP {response.status_code}: {response.text[:300]}"
    message = error.get("message") or f"HTTP {response.status_code}"
    if error.get("details"):
        message += f" ({error['details']})"
    return error.get("code"), message


def get_json(path, **params):
    response = api("GET", path, params=params)
    response.raise_for_status()
    return response.json()


def get_all(path, **params):
    """Fetch every page of a paginated list."""
    items, page = [], 1
    while True:
        data = get_json(path, page=page, limit=100, **params)
        items += data["results"]
        if not data["results"] or len(items) >= data["total"]:
            return items
        page += 1

Run the next cell to check the connection. It lists the artifact types in your organization; an authentication error here means the API key, secret or host is wrong.

In [ ]:
artifact_types = [t for t in get_json("/artifact-types") if not t["is_disabled"]]
pd.DataFrame(artifact_types)[["name", "slug", "tag"]]

<a id='toc5__'></a>

## Choose what to upload

<a id='toc5_1__'></a>

### Choose the artifact type

Every row in your file becomes an artifact of the same type. Set `ARTIFACT_TYPE` in [Configure your upload](#toc2__) to a `name` or `slug` from the table above. To upload several types, use one file per type.

The next cell also lists the statuses this type offers. A row with no `status` starts in the type's default status, `open`.

In [ ]:
artifact_type = next(
    (t for t in artifact_types if ARTIFACT_TYPE.strip().lower() in (t["slug"], t["name"].lower())),
    None,
)
if artifact_type is None:
    raise ValueError(f"No artifact type named {ARTIFACT_TYPE!r}. Use a name or slug from the table above.")
ARTIFACT_TYPE_CUID = artifact_type["cuid"]
print(f"Uploading {artifact_type['name']!r} artifacts. Statuses:", ", ".join(s["name"] for s in artifact_type["statuses"]))

<a id='toc5_2__'></a>

### Review the fields for this artifact type

These are the custom fields you can fill in from your file. A column in your file is matched to a field by its **key** or its **title**, ignoring case.

Fields marked `required` must have a value in every row, or the artifact is rejected. Attachment fields and calculated fields cannot be set from a file.

Each artifact type can also turn built-in fields on or off, and require some of them when an artifact is created. That setting is not available through the API, so the dry run does not check it: a row that leaves out a required built-in field, or fills in one the type has turned off, fails at upload with a message naming the field. In particular, **risk area is required by every artifact type that uses it**, which by default is most of them, so fill in `risk_area` unless you know your type has turned it off.

In [ ]:
UNSUPPORTED_TYPES = ("array:attachments", "string:code-python", "aggregate")
CORE_COLUMNS = ["record", "title", "description", "severity", "status", "assignee", "risk_area", "due_date"]

# A field with no artifact types listed applies to every type.
fields = [f for f in get_json("/artifact-fields") if ARTIFACT_TYPE_CUID in (f.get("artifact_type_cuids") or [ARTIFACT_TYPE_CUID])]
fields_by_key = {f["key"]: f for f in fields}


def allowed_values(field):
    return (field.get("arrayOptions") or {}).get("items") or []


pd.DataFrame(
    [
        {
            "title": f["title"],
            "key": f["key"],
            "type": f["type"],
            "required": bool(f.get("requiredOnRegistration")),
            "allowed values": ", ".join(allowed_values(f)),
        }
        for f in fields
        if not f["type"].startswith(UNSUPPORTED_TYPES)
    ],
    columns=["title", "key", "type", "required", "allowed values"],
)

<a id='toc6__'></a>

## Prepare your file

<a id='toc6_1__'></a>

### Load your file

Your file needs one row per artifact and a header row. `record`, `title` and `description` are always required. Other columns are optional unless your organization requires them (see the fields above). The columns you can use are:

| Column | What goes in it |
|---|---|
| `record` | The name of the record the artifact belongs to, exactly as it appears in your inventory. Always required. |
| `title` | A short name for the artifact. Always required. |
| `description` | The full details of the artifact. Always required. |
| `severity` | The name of a severity in your organization, such as `High`. |
| `status` | The name of one of this artifact type's statuses. Leave it out to start in the default status. |
| `assignee` | The email address of the user responsible for the artifact. They must be a member of the record's group. |
| `risk_area` | The name of an existing risk area, such as `Data Quality`. Required by most artifact types. |
| `due_date` | The date by which the artifact should be resolved. |
| *custom field* | A field from the table above, as its key or title. |

Values in a list (multiple choice fields, user fields) are separated by `;`. Dates can be real Excel dates or text in any common format, and are converted to `YYYY-MM-DD`. Text dates such as `02/03/2024` are read month first, as 3 February 2024. If your file puts the day first, set `DATES_DAY_FIRST = True` in [Configure your upload](#toc2__); real Excel dates and `YYYY-MM-DD` text are read correctly either way. Date and time values without a time zone are read as UTC. Yes/no fields accept `yes`, `no`, `true`, `false`, `1` or `0`.

This notebook comes with `sample_artifacts.csv` and `sample_artifacts.xlsx` to start from. Their record names are examples: replace them with records from your inventory, or the dry run reports them as problems. Set `FILE_PATH` in [Configure your upload](#toc2__) to your own file.

In [ ]:
read = pd.read_excel if FILE_PATH.lower().endswith(".xlsx") else pd.read_csv
rows = read(FILE_PATH, dtype=str).fillna("")
rows.columns = [str(c).strip() for c in rows.columns]
# The error column of a failed_rows.csv from an earlier run is not uploaded.
rows = rows.drop(columns="error", errors="ignore")
print(f"{len(rows)} rows, columns: {list(rows.columns)}")
rows.head()

<a id='toc6_2__'></a>

### Map columns to fields

The next cell works out what each column in your file means. If a column header does not match, add it to `COLUMN_MAP` in [Configure your upload](#toc2__), for example `{"Model Name": "record", "Issue": "title", "Details": "description", "Owner": "assignee"}`. Columns that still do not match are ignored and listed so you can check none were missed.

In [ ]:
fields_by_name = {}
for f in fields:
    fields_by_name[f["key"].lower()] = f
    fields_by_name[f["title"].lower()] = f


def target_for(column):
    """Return what a column maps to: ("core", name) or ("field", key)."""
    column = COLUMN_MAP.get(column, column).strip()
    if column.lower().replace(" ", "_") in CORE_COLUMNS:
        return ("core", column.lower().replace(" ", "_"))
    field = fields_by_name.get(column.lower())
    if field:
        return ("field", field["key"])
    return None


mapping = {column: target_for(column) for column in rows.columns}
unsupported = [c for c, t in mapping.items() if t and t[0] == "field" and fields_by_key[t[1]]["type"].startswith(UNSUPPORTED_TYPES)]
unmapped = [c for c, t in mapping.items() if t is None]

for required in ("record", "title", "description"):
    if ("core", required) not in mapping.values():
        raise ValueError(f"Your file needs a {required!r} column (or map one with COLUMN_MAP).")

print("Unmapped columns, which will be ignored:", unmapped or "none")
print("Columns for fields that cannot be set from a file, which will be ignored:", unsupported or "none")
pd.DataFrame([{"column": c, "maps to": f"{t[0]}: {t[1]}"} for c, t in mapping.items() if t and c not in unsupported])

<a id='toc6_3__'></a>

### Convert rows into artifacts

The next cell turns each row into the request the API expects. Record names, severities, statuses, risk areas and email addresses are looked up in your organization and replaced with their ValidMind identifiers, so it only fetches the lists your file actually uses. Unless the API key's user is an administrator, it can only see records and users in the groups it belongs to, so a record or user outside those groups is reported as not found.

In [ ]:
used = {t for t in mapping.values() if t}
used_types = {fields_by_key[k]["type"] for kind, k in used if kind == "field"}


def by_name(items, key="name"):
    return {item[key].strip().lower(): item["cuid"] for item in items}


records_by_name = by_name(get_all("/models", columns="name,cuid"))
record_cuids = set(records_by_name.values())
lookups = {
    "severity": {s["name"].strip().lower(): s["level"] for s in get_json("/artifacts/severities")["data"]} if ("core", "severity") in used else {},
    "status": by_name(artifact_type["statuses"]),
    "risk_area": by_name(get_json("/risk-areas")) if ("core", "risk_area") in used else {},
}
needs_users = ("core", "assignee") in used or "array:model-user" in used_types
users_by_email = by_name(get_json("/organization/users")["users"], key="email") if needs_users else {}


def split_list(value):
    return [v.strip() for v in value.split(";") if v.strip()]


def user_cuid(email):
    cuid = users_by_email.get(email.lower())
    if not cuid:
        raise ValueError(f"no user with email {email!r} that the API key's user can see")
    return cuid


def record_cuid(value):
    """Accept a record's name or its ValidMind identifier."""
    if value in record_cuids:
        return value
    cuid = records_by_name.get(value.lower())
    if not cuid:
        raise ValueError(f"no record named {value!r} that the API key's user can see")
    return cuid

In [ ]:
def yes_no(value):
    answer = value.strip().lower()
    if answer not in ("yes", "no", "true", "false", "1", "0"):
        raise ValueError(f"{value!r} is not yes or no")
    return answer in ("yes", "true", "1")


def parse_date(value):
    """Parse a date. DATES_DAY_FIRST applies only to text like 02/03/2024, never to
    YYYY-MM-DD, which is also how real Excel dates arrive."""
    is_iso = re.match(r"\d{4}-\d{1,2}-\d{1,2}", value)
    return pd.to_datetime(value, dayfirst=DATES_DAY_FIRST and not is_iso)


def convert(value, field):
    """Convert a spreadsheet value into the type the field stores."""
    kind = field["type"]
    if kind == "array:model-user":
        emails = split_list(value)
        if len(emails) > 1 and not (field.get("userOptions") or {}).get("many"):
            raise ValueError("this field takes one user")
        return [user_cuid(email) for email in emails]
    if kind.startswith("array:"):
        return split_list(value)
    if kind == "number":
        number = float(value.replace(",", ""))
        places = (field.get("numberOptions") or {}).get("decimalPlaces")
        return round(number, places) if places is not None else number
    if kind.startswith("boolean"):
        return yes_no(value)
    if kind == "string:date":
        return parse_date(value).date().isoformat()
    if kind == "string:date-time":
        # Stored as epoch milliseconds, like the ValidMind Platform; naive values are UTC.
        return str(int(parse_date(value).timestamp() * 1000))
    return value


def to_artifact(row):
    """Build the create request for one row. Returns (record cuid, payload, problems, notes)."""
    record, payload = None, {"custom_fields": {}}
    problems, notes = [], []
    for column, target in mapping.items():
        value = str(row[column]).strip()
        if not target or not value or column in unsupported:
            continue
        kind, key = target
        try:
            if kind == "core" and key == "record":
                record = record_cuid(value)
            elif kind == "core" and key in ("title", "description"):
                payload[key] = value
            elif kind == "core" and key in lookups:
                if value.lower() not in lookups[key]:
                    raise ValueError(f"no {key.replace('_', ' ')} named {value!r}")
                payload["severity" if key == "severity" else f"{key}_cuid"] = lookups[key][value.lower()]
            elif kind == "core" and key == "assignee":
                payload["assignee_cuid"] = user_cuid(value)
            elif kind == "core" and key == "due_date":
                payload["due_at"] = parse_date(value).date().isoformat() + "T00:00:00Z"
            else:
                field = fields_by_key[key]
                payload["custom_fields"][key] = convert(value, field)
                outside = [v for v in split_list(value) if v not in allowed_values(field)] if allowed_values(field) else []
                if outside:
                    notes.append(f"{column}: {', '.join(outside)} not in the allowed values (accepted only if the field allows other values)")
        except ValueError as e:
            problems.append(f"{column}: {e}")
    for column, target in mapping.items():
        if target in (("core", "record"), ("core", "title"), ("core", "description")) and not str(row[column]).strip():
            problems.append(f"{target[1]} is empty")
    for field in fields:
        if field.get("requiredOnRegistration") and field["key"] not in payload["custom_fields"]:
            problems.append(f"{field['title']} is required but empty")
    return record, payload, problems, notes

<a id='toc7__'></a>

## Check your file with a dry run

The dry run checks every row and creates nothing. Each row gets a status:

- `ready` — the row will be uploaded.
- `skip: artifact already exists` — the record already has an artifact of this type with the same title, or the same artifact appears earlier in your file. The upload skips it.
- `problem` — the row cannot be uploaded as it is, for example because a required field is empty or a record or user does not exist. Fix it in your file and load it again.

`notes` are worth reading but do not stop a row: the API makes the final decision when it is uploaded.

The check for existing artifacts compares titles ignoring case, and only sees artifacts on records the API key's user can access.

In [ ]:
def all_artifacts(type_cuid):
    """Fetch every artifact of one type, a page at a time."""
    items, page = [], 1
    while True:
        data = get_json(f"/artifacts/{type_cuid}", page=page, limit=100)
        items += [item["attributes"] for item in data["data"]]
        if page >= (data["meta"]["pages"] or 1):
            return items
        page += 1


existing = {(a["inventory_model"]["cuid"], a["title"].strip().lower()) for a in all_artifacts(ARTIFACT_TYPE_CUID)}

checks, seen = [], set()
for index, row in rows.iterrows():
    record, payload, problems, notes = to_artifact(row)
    key = (record, payload.get("title", "").lower())
    if problems:
        status = "problem"
    elif key in existing or key in seen:
        status = "skip: artifact already exists"
    else:
        status = "ready"
    if status == "ready":
        seen.add(key)
    checks.append(
        {"row": index + 2, "title": payload.get("title", ""), "status": status, "problems": "; ".join(problems), "notes": "; ".join(notes), "record": record, "payload": payload}
    )

checks = pd.DataFrame(checks)
print(checks["status"].value_counts().to_string())
if unmapped or unsupported:
    print()
    print(f"Warning: these columns are not uploaded: {unmapped + unsupported}. See 'Map columns to fields'.")
checks.drop(columns=["record", "payload"])

<a id='toc8__'></a>

## Upload your artifacts

When the dry run looks right, set `DRY_RUN = False` in [Configure your upload](#toc2__) and run the notebook again. Only `ready` rows are sent.

By default, ValidMind does not email anyone about each new artifact, so a large upload does not flood inboxes. Set `SKIP_EMAIL_NOTIFICATIONS = False` in [Configure your upload](#toc2__) to send those emails.

Each new artifact also starts any workflow your organization runs when an artifact of this type is created, exactly as when it is logged in the ValidMind Platform. There is no setting to skip it, so check with whoever owns that workflow before uploading many artifacts at once.

Some rows can only be checked by the API when they are uploaded, and are reported as `failed` with its message:

- a required built-in field is empty, or the type has turned off a field the row fills in
- the artifact type cannot be used on that kind of record
- the assignee is not a member of the record's group

In [ ]:
to_upload = checks[checks["status"] == "ready"]
if DRY_RUN:
    print(f"Dry run: {len(to_upload)} rows would be uploaded. Set DRY_RUN = False at the top of the notebook to upload them.")
    to_upload = to_upload.iloc[:0]


def upload(check):
    """Create the artifact for one row and return its result."""
    result = {"row": check.row, "title": check.title, "status": "failed", "artifact": None, "error": ""}
    try:
        response = api(
            "POST",
            f"/models/{check.record}/artifacts/{ARTIFACT_TYPE_CUID}",
            params={"skip_email_notification": str(SKIP_EMAIL_NOTIFICATIONS).lower()},
            json=check.payload,
        )
    except requests.RequestException as e:
        return {**result, "error": f"request failed: {e}"}
    if not response.ok:
        return {**result, "error": api_error(response)[1]}
    return {**result, "status": "created", "artifact": response.json().get("cuid")}


results, seen_errors = [], set()
for count, check in enumerate(to_upload.itertuples(), 1):
    result = upload(check)
    results.append(result)
    if result["status"] == "failed" and result["error"] not in seen_errors:
        seen_errors.add(result["error"])
        print(f"Row {check.row} failed: {result['error']}")
    if count % 25 == 0:
        print(f"{count} of {len(to_upload)} rows processed")

<a id='toc8_1__'></a>

### Review the results

The table shows what happened to every row of your file, including rows the dry run did not send. After a dry run, rows that would be uploaded show as `ready (not uploaded)`. Rows that failed or had a problem are written to `failed_rows.csv`, in the same folder as your file, with your original columns and an `error` column. Fix them there and point `FILE_PATH` at that file to upload them; its `error` column is ignored, and artifacts that were created will be skipped if you include them again. When no row fails, a `failed_rows.csv` left by an earlier run is deleted.

In [ ]:
report = checks[["row", "title", "status"]].assign(artifact="", error=checks["problems"])
if DRY_RUN:
    report["status"] = report["status"].replace("ready", "ready (not uploaded)")
for result in results:
    report.loc[report["row"] == result["row"], ["status", "artifact", "error"]] = [result["status"], result["artifact"] or "", result["error"]]
print(report["status"].value_counts().to_string())

failed = report[report["status"].isin(["failed", "problem"])]
failed_path = os.path.join(os.path.dirname(os.path.abspath(FILE_PATH)), "failed_rows.csv")
if len(failed):
    rows.loc[failed["row"] - 2].assign(error=failed["error"].values).to_csv(failed_path, index=False)
    print(f"{len(failed)} rows written to {failed_path}")
elif os.path.exists(failed_path) and failed_path != os.path.abspath(FILE_PATH):
    os.remove(failed_path)
    print(f"No rows failed; removed {failed_path} from an earlier run")
report

<a id='toc9__'></a>

## In summary

In this notebook, you learned how to:

- [x] Connect to the ValidMind REST API with your API key and secret
- [x] Review the fields an artifact type expects
- [x] Map the columns of an Excel or CSV file to artifact fields
- [x] Check a file with a dry run before creating anything
- [x] Create artifacts in bulk and review what happened to each row

<a id='toc10__'></a>

## Next steps

Open a record in the ValidMind Platform to see its new artifacts, or the artifacts list to see them across your inventory. From there you can assign them, add attachments, and move them through their statuses.

<a id='toc10_1__'></a>

### Discover more learning resources

- [ValidMind REST API](https://docs.validmind.ai/reference/validmind-rest-api-vm.html) — every endpoint this notebook uses, and many more
- [Bulk upload inventory records](bulk_upload_inventory_records.ipynb) — the companion notebook for loading records

Or, visit our [documentation](https://docs.validmind.ai/) to learn more about ValidMind.

<!-- VALIDMIND COPYRIGHT -->

<small>

***

Copyright © 2023-2026 ValidMind Inc. All rights reserved.<br>
Refer to [LICENSE](https://github.com/validmind/validmind-library/blob/main/LICENSE) for details.<br>
SPDX-License-Identifier: AGPL-3.0 AND ValidMind Commercial</small>